# este notebook preprocesa las descripciones para ser usado en uno posterior

In [ ]:
import pandas as pd 

from transformers import pipeline

In [ ]:
train = pd.read_csv('../input/petfinder-adoption-prediction/train/train.csv').set_index("PetID")
test = pd.read_csv('../input/petfinder-adoption-prediction/test/test.csv').set_index("PetID")

In [ ]:
import os, requests
os.makedirs("/tmp/emo", exist_ok=True)
url = "https://huggingface.co/j-hartmann/emotion-english-distilroberta-base/resolve/main/"
archivos = ["config.json", "pytorch_model.bin", "vocab.json", "merges.txt",
"tokenizer_config.json", "special_tokens_map.json"]
for f in archivos:
    open("/tmp/emo/" + f, "wb").write(requests.get(url + f).content)
classifier = pipeline("text-classification", model="/tmp/emo",
tokenizer="/tmp/emo", return_all_scores=True, device=0)

descr = classifier(train.Description.fillna("empty").astype(str).values.tolist(), max_length=512, truncation=True)
descr = pd.DataFrame([{r["label"]: r["score"] for r in res} for res in descr], index=train.index)
descr

In [ ]:
descr_test = classifier(test.Description.fillna("empty").astype(str).values.tolist(), max_length=512, truncation=True)
descr_test = pd.DataFrame([{r["label"]: r["score"] for r in res} for res in descr_test], index=test.index)
descr_test

In [ ]:
descr.to_parquet("train.parquet")
descr_test.to_parquet("test.parquet")